In [1]:
import itertools as it

for a, b in it.product(range(3), repeat=2):
    if a != 0: continue
    print((a, b))

(0, 0)
(0, 1)
(0, 2)


In [ ]:
from ansatz import Ansatz, hdict_builder, restrict
import itertools as it
import sympy as sp

M_OF = [1, 0, -1]
BAR = [2, 1, 0]

def family(name):
    extra = []
    if name == 'A':         hd = hdict_builder('hermitian')
    elif name == 'B':       hd = hdict_builder('dipolar_real')
    elif name == 'Bp':      hd = hdict_builder('dipolar')
    elif name == 'B_R':     hd = hdict_builder('dipolar_real', reflection=True)
    elif name == 'B_RZ2':
        hd = hdict_builder('dipolar_real', reflection=True)
        def sym(a, b, l):
            return sp.Symbol(f'S_{min(a, b)}{max(a, b)}^{l}', real=True)
        can_keys = set()
        for a, b in it.product(range(3), repeat=2):
            a2, b2 = BAR[a], BAR[b]
            k1, k2 = (min(a, b), max(a, b)), (min(a2, b2), max(a2, b2))
            if (k1 in can_keys) or (k2 in can_keys):
                continue
            can_keys.add(k1)
            extra += [sym(a, b, '0') - sym(a2, b2, '0'), sym(a, b, 'x') - sym(a2, b2, 'x'), sym(a, b, 'z') + sym(a2, b2, 'z')]
    elif name == 'C':       hd = hdict_builder('real')
    elif name == 'D':       hd = hdict_builder('real', reflection=True)
    elif name == 'E':
        def pattern_E(a, b): return int(M_OF[a] + M_OF[b] == 0)
        hd = hdict_builder('real', pattern=pattern_E)
    elif name == 'E_B':
        def pattern_E(a, b): return int(M_OF[a] + M_OF[b] == 0)
        hd = hdict_builder('dipolar_real', pattern=pattern_E)
    elif name == 'Z3':
        # a can be fixed to 0, changing it to any other value would yield redundant equations
        hd = hdict_builder('hermitian')
        a = 0
        for b in range(3):
            M = hd[(a, b)]
            for k in (1, 2):
                ak, bk = (a+k)%3, (b+k)%3
                Mk = hd[ak, bk]
                for i, j in it.product(range(3), repeat=2):
                    e = sp.expand(Mk[(i+k)%3, (j+k)%3] - M[i, j])
                    extra += [sp.re(e), sp.im(e)]
    elif name == 'Z3_RR':
        hd = hdict_builder('real', reflection=True)
        a = 0
        for b in range(3):
            M = hd[(a, b)]
            for k in (1, 2):
                Mk = hd[(a+k)%3, (b+k)%3]
                for i, j in it.product(range(3), repeat=2):
                    e = sp.expand(Mk[(i+k)%3, (j+k)%3] - M[i, j])
                    extra += [sp.re(e), sp.im(e)]
    else:
        raise ValueError(f'symmetry family {name} is not defined')
    extra = [e for e in extra if e != 0]
    ans = Ansatz(hd, name=name)
    return restrict(ans, extra, name) if extra else ans

In [9]:
M_OF = [1, 0, -1]
pattern=lambda a, b: int(M_OF[a] + M_OF[b] == 0)
def pattern_self(a, b):
    return int(M_OF[a] + M_OF[b] == 0)
print(pattern(0, 2))
print(pattern_self(2, 0))

1
1


In [ ]:
def rebuild_family(name, syms):
    